# Chapter 30 — Neural Networks from Scratch

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

digits = load_digits()
X, y = digits.data / 16.0, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                      stratify=y, random_state=0)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# A single neuron is a dot product plus a nonlinearity. Nothing more.
r = np.random.default_rng(30)
w = r.normal(size=64)
b = 0.0
x = X[0]                              # one digit, 64 pixel values

z = x @ w + b                         # the weighted sum from Chapter 9
a = max(0, z)                         # ReLU: pass positive, zero out negative

print(f"pixel vector shape: {x.shape}")
print(f"weighted sum z = {z:.3f}")
print(f"after ReLU,  a = {a:.3f}")
print(f"\nthat is the entire computation one neuron performs.")
print(f"a layer is many of these run in parallel; a network is")
print(f"layers of them run in sequence.")

### Block 2  (`c2.py`)

In [ ]:
# Without a nonlinearity, stacking layers is pointless: two linear layers
# collapse into one. The nonlinearity is what makes depth mean anything.
from sklearn.datasets import make_circles
Xc, yc = make_circles(n_samples=400, noise=0.08, factor=0.4, random_state=0)

# A "network" with no activation: z2 = (z1 @ W2) = ((x @ W1) @ W2) = x @ (W1 @ W2)
# which is just one big matrix -- a linear model wearing a costume.
r2 = np.random.default_rng(30)
W1c = r2.normal(size=(2, 8)); W2c = r2.normal(size=(8, 1))
combined = W1c @ W2c
print(f"W1 shape {W1c.shape}, W2 shape {W2c.shape}")
print(f"W1 @ W2 shape {combined.shape}  <- collapses to one 2x1 matrix")
print("two linear layers ARE one linear layer, just slower to compute.")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
lin = LogisticRegression().fit(Xc, yc)
print(f"\nlinear model on a ring inside a ring: "
      f"{accuracy_score(yc, lin.predict(Xc)):.3f} accuracy")
print("no straight line separates a ring from its centre, so a stack of")
print("linear layers scores exactly what guessing scores. This is why")
print("ReLU exists: it lets each layer bend the boundary, not just")
print("rotate it.")

### Block 3  (`c3.py`)

In [ ]:
# The forward pass through a real two-layer network, on one digit.
D_in, H, D_out = 64, 32, 10
r3 = np.random.default_rng(30)
W1 = r3.normal(0, np.sqrt(2/D_in), (D_in, H))    # He initialization
b1 = np.zeros(H)
W2 = r3.normal(0, np.sqrt(2/H), (H, D_out))
b2 = np.zeros(D_out)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)          # overflow guard
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def forward(Xb):
    z1 = Xb @ W1 + b1
    a1 = np.maximum(0, z1)                         # ReLU
    z2 = a1 @ W2 + b2
    p = softmax(z2)
    return z1, a1, z2, p

x = X[:1]
z1, a1, z2, p = forward(x)
print(f"input           {x.shape}")
print(f"after layer 1   {a1.shape}   ({(a1 > 0).sum()} of {H} neurons fired)")
print(f"after layer 2   {z2.shape}")
print(f"after softmax   {p.shape}   sums to {p.sum():.6f}")
print(f"\npredicted digit: {p.argmax()}   true label: {y[:1][0]}")
print(f"confidence in that digit: {p.max():.3f}")

### Block 4  (`c4.py`)

In [ ]:
# Backpropagation is the chain rule from Chapter 10, applied layer by
# layer. Each gradient below is checked against Chapter 10's numerical
# method before it is trusted.
Y = np.eye(10)[y[:5]]                 # one-hot targets, 5 examples
Xb = X[:5]

def loss_and_grads(Xb, Y):
    z1, a1, z2, p = forward(Xb)
    n = len(Xb)
    loss = -np.sum(Y * np.log(p + 1e-12)) / n

    dz2 = (p - Y) / n                              # softmax + cross-entropy
    dW2 = a1.T @ dz2
    db2 = dz2.sum(0)
    da1 = dz2 @ W2.T
    dz1 = da1 * (z1 > 0)                            # ReLU gradient
    dW1 = Xb.T @ dz1
    db1 = dz1.sum(0)
    return loss, (dW1, db1, dW2, db2)

loss, (dW1, db1, dW2, db2) = loss_and_grads(Xb, Y)

# numerical check on a handful of W1 entries, exactly Chapter 10's method
eps = 1e-5
checks = [(43, 7), (27, 21), (43, 16)]
print(f"{'entry':>10}{'analytic':>12}{'numerical':>12}{'match':>8}")
for i, j in checks:
    orig = W1[i, j]
    W1[i, j] = orig + eps
    lp, _ = loss_and_grads(Xb, Y)
    W1[i, j] = orig - eps
    lm, _ = loss_and_grads(Xb, Y)
    W1[i, j] = orig
    numeric = (lp - lm) / (2 * eps)
    match = abs(numeric - dW1[i, j]) < 1e-4
    print(f"({i:>2},{j:>2}){dW1[i,j]:>12.6f}{numeric:>12.6f}{str(match):>8}")

print(f"\nstarting loss on these five examples: {loss:.4f}")
print(f"(a fresh, untrained network on a 10-class problem should sit")
print(f" near ln(10) = {np.log(10):.4f})")

### Block 5  (`c5.py`)

In [ ]:
# Train on the real data: forward, backward, update, repeat.
def forward_train(Xb, W1, b1, W2, b2):
    z1 = Xb @ W1 + b1
    a1 = np.maximum(0, z1)
    z2 = a1 @ W2 + b2
    return z1, a1, softmax(z2)

D_in, H, D_out = 64, 32, 10
r5 = np.random.default_rng(30)
W1t = r5.normal(0, np.sqrt(2/D_in), (D_in, H)); b1t = np.zeros(H)
W2t = r5.normal(0, np.sqrt(2/H), (H, D_out));    b2t = np.zeros(D_out)
Ytr = np.eye(10)[ytr]
eta = 0.5

print(f"{'epoch':>7}{'train loss':>13}{'test accuracy':>15}")
for epoch in range(401):
    z1, a1, p = forward_train(Xtr, W1t, b1t, W2t, b2t)
    loss = -np.sum(Ytr * np.log(p + 1e-12)) / len(Xtr)

    dz2 = (p - Ytr) / len(Xtr)
    dW2, db2 = a1.T @ dz2, dz2.sum(0)
    dz1 = (dz2 @ W2t.T) * (z1 > 0)
    dW1, db1 = Xtr.T @ dz1, dz1.sum(0)
    W1t -= eta * dW1; b1t -= eta * db1
    W2t -= eta * dW2; b2t -= eta * db2

    if epoch % 100 == 0:
        _, _, pte = forward_train(Xte, W1t, b1t, W2t, b2t)
        acc = (pte.argmax(1) == yte).mean()
        print(f"{epoch:>7}{loss:>13.4f}{acc:>15.4f}")

_, _, pte = forward_train(Xte, W1t, b1t, W2t, b2t)
final_acc = (pte.argmax(1) == yte).mean()
print(f"\nfinal test accuracy: {final_acc:.4f}")

### Block 6  (`c6.py`)

In [ ]:
# The baseline discipline from Chapter 1: before trusting the network,
# check what a much simpler model achieves on the same split.
baseline = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=2000))
baseline.fit(Xtr, ytr)
base_acc = baseline.score(Xte, yte)

print(f"logistic regression baseline:  {base_acc:.4f}")
print(f"hand-built network:            {final_acc:.4f}")
print(f"gain over the baseline:        {final_acc - base_acc:+.4f}")

# where does the network still fail?
_, _, pte = forward_train(Xte, W1t, b1t, W2t, b2t)
pred = pte.argmax(1)
wrong = np.where(pred != yte)[0]
print(f"\n{len(wrong)} of {len(yte)} test digits misclassified")
print(f"{'true':>6}{'predicted':>11}{'confidence':>13}")
for i in wrong[:6]:
    print(f"{yte[i]:>6}{pred[i]:>11}{pte[i, pred[i]]:>13.3f}")